In [ ]:
import os
import time
import scvi
import psutil
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.metrics import accuracy_score




def train_scanvi(train_data, test_data, label_name):
    """
    使用 SCANVI 训练模型并计算测试集准确性，同时记录运行时间和内存使用情况。
    
    参数:
    - train_data: 训练集 (AnnData 对象)
    - test_data: 测试集 (AnnData 对象)
    - label_name: 标签列名

    
    返回:
    - test_accuracy: 测试集的准确性

    - runtime: 运行时间（秒）
    - memory_usage_gb: 内存使用（GB）
    - predicted_labels_df: 每个 cell 的预测标签 (DataFrame 格式)
    """
    # 记录开始时间和初始内存使用

    start_time = time.time()
    process = psutil.Process(os.getpid())
    initial_memory = process.memory_info().rss / (1024 ** 3)  # 初始内存使用 (GB)

    # 设置随机种子

    scvi.settings.seed = 114514

    # 设置 SCANVI 模型并训练

    scvi.model.SCANVI.setup_anndata(train_data, labels_key=label_name, unlabeled_category="Unknown")
    scanvi_model = scvi.model.SCANVI(train_data)
    scanvi_model.train(max_epochs=50)  # 可调整 max_epochs 参数

    # 对测试集进行预测

    predicted_labels = scanvi_model.predict(test_data)
    true_labels = test_data.obs[label_name].values

    # 计算测试集准确性

    test_accuracy = accuracy_score(true_labels, predicted_labels)

    # 创建预测标签的 DataFrame

    predicted_labels_df = pd.DataFrame({
        "cell_id": test_data.obs_names,
        "true_label": true_labels,
        "predicted_label": predicted_labels

    })

    # 记录结束时间和最终内存使用

    end_time = time.time()
    final_memory = process.memory_info().rss / (1024 ** 3)  # 最终内存使用 (GB)

    # 计算运行时间和内存使用

    runtime = end_time - start_time

    memory_usage_gb = final_memory - initial_memory

    return test_accuracy, runtime, memory_usage_gb, predicted_labels_df

In [58]:
split_data_path = "/data/jiangjunyao/AEGAS data/celltype annotation/AEGAS_anno/intra_fivefold_split/"
outdir = '/data/jiangjunyao/AEGAS data/celltype annotation/anno_result/predict_result/scanvi_'
h5ad_files = [f for f in os.listdir(split_data_path) if os.path.isdir(os.path.join(split_data_path, f))]

output_results = []

# 遍历每个数据集

for dataset in h5ad_files:
    print(f"处理数据集：{dataset}")
    dataset_path = split_data_path +dataset

    fold_accuracies = []
    fold_runtimes = []
    fold_memories = []

    # 读取训练集和测试集
    train_data = sc.read_h5ad(dataset_path+"/train.h5ad")
    test_data = sc.read_h5ad(dataset_path+"/test.h5ad")

    # 确保数据中有标签列

    label_name = "celltype"  # 假设标签列为 "cell_type"，请根据实际情况修改

    if label_name not in train_data.obs.columns or label_name not in test_data.obs.columns:
        raise ValueError(f"missing label:  '{label_name}'")

    # 使用 scANVI 训练并计算测试准确性、运行时间和内存使用

    test_accuracy, runtime, memory_usage_gb,result = train_scanvi(train_data, test_data, label_name)
    fold_accuracies.append(test_accuracy)
    fold_runtimes.append(runtime)
    fold_memories.append(memory_usage_gb)
    result.to_csv(outdir+dataset+'.csv')

    # 保存每个数据集的结果

    output_results.append({
        "dataset": dataset,
        "fold_accuracies": fold_accuracies,
        "fold_runtimes": fold_runtimes,
        "fold_memories": fold_memories,
        "mean_accuracy": np.mean(fold_accuracies),
        "mean_runtime": np.mean(fold_runtimes),
        "mean_memory": np.mean(fold_memories)
    })

# 创建 DataFrame 并保存结果

results_df = pd.DataFrame(output_results)
results_df = results_df[["dataset", "mean_accuracy", "mean_runtime", "mean_memory"]]  # 选择关键列

print(results_df)


处理数据集：ProksNM_11_mouseembryo_fold_1


Seed set to 114514


INFO     Training for 50 epochs.                                                                                   


/data/jiangjunyao/miniconda3/envs/normal/lib/python3.9/site-packages/scvi/data/fields/_base_field.py:64: UserWarning: adata.X does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/data/jiangjunyao/miniconda3/envs/normal/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in 

Epoch 1/50:   0%|          | 0/50 [00:00<?, ?it/s]

...

/data/jiangjunyao/miniconda3/envs/normal/lib/python3.9/site-packages/scvi/module/_scanvae.py:304: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -px.log_prob(x).sum(-1)
/data/jiangjunyao/miniconda3/envs/normal/lib/python3.9/site-packages/scvi/module/_scanvae.py:304: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -px.log_prob(x).sum(-1)


Epoch 50/50: 100%|██████████| 50/50 [00:11<00:00,  4.54it/s, v_num=1, train_loss_step=353, train_loss_epoch=277]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [00:11<00:00,  4.54it/s, v_num=1, train_loss_step=353, train_loss_epoch=277]
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
                         dataset  mean_accuracy  mean_runtime  mean_memory
0  ProksNM_11_mouseembryo_fold_1       0.925187      9.919697    -5.769333
1  ProksNM_11_mouseembryo_fold_2       0.917706      9.810640     0.000000
2  ProksNM_11_mouseembryo_fold_3       0.912718      9.794858     0.000061
3  ProksNM_11_mouseembryo_fold_4       0.915212      9.007106     0.000000
4  ProksNM_11_mouseembryo_fold_5       0.917500      8.297318     0.000969
5  ProksNM_12_humanembryo_fold_1       0.709677      9.411921     0.000172
6  ProksNM_12_humanembryo_fold_2       0.703226      9.403970     0.000225
7  ProksNM_12_humanembryo_fold_3       0.677419      9.371180     0.000004
8  ProksNM_12_humanembryo_fold_4       0.700431     10.747179     0.000221
9  ProksNM_12_humanem

/data/jiangjunyao/miniconda3/envs/normal/lib/python3.9/site-packages/scvi/data/fields/_base_field.py:64: UserWarning: adata.X does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/data/jiangjunyao/miniconda3/envs/normal/lib/python3.9/site-packages/scvi/data/fields/_dataframe_field.py:224: UserWarning: Category 0 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/data/jiangjunyao/miniconda3/envs/normal/lib/python3.9/site-packages/scvi/data/fields/_scanvi.py:56: UserWarning: Category 0 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  mapping = _make_column_categorical(


In [59]:
results_df.to_csv('/data/jiangjunyao/AEGAS data/celltype annotation/AEGAS_anno/anno_summary/scanvi_intra2.csv')

In [ ]:
split_data_path = "/data/jiangjunyao/AEGAS data/celltype annotation/AEGAS_anno/intra_fivefold_split/"
outdir = '/data/jiangjunyao/AEGAS data/celltype annotation/anno_result/predict_result/scanvi_'
h5ad_files = [f for f in os.listdir(split_data_path) if os.path.isdir(os.path.join(split_data_path, f))]

output_results = []

# 遍历每个数据集

for dataset in h5ad_files:
    print(f"处理数据集：{dataset}")
    dataset_path = split_data_path +dataset

    fold_accuracies = []
    fold_runtimes = []
    fold_memories = []

    # 读取训练集和测试集
    train_data = sc.read_h5ad(dataset_path+"/train.h5ad")
    test_data = sc.read_h5ad(dataset_path+"/test.h5ad")

    # 确保数据中有标签列

    label_name = "celltype"  # 假设标签列为 "cell_type"，请根据实际情况修改

    if label_name not in train_data.obs.columns or label_name not in test_data.obs.columns:
        raise ValueError(f"missing label:  '{label_name}'")

    # 使用 scANVI 训练并计算测试准确性、运行时间和内存使用

    test_accuracy, runtime, memory_usage_gb,result = train_scanvi(train_data, test_data, label_name)
    fold_accuracies.append(test_accuracy)
    fold_runtimes.append(runtime)
    fold_memories.append(memory_usage_gb)
    result.to_csv(outdir+dataset+'.csv')

    # 保存每个数据集的结果

    output_results.append({
        "dataset": dataset,
        "fold_accuracies": fold_accuracies,
        "fold_runtimes": fold_runtimes,
        "fold_memories": fold_memories,
        "mean_accuracy": np.mean(fold_accuracies),
        "mean_runtime": np.mean(fold_runtimes),
        "mean_memory": np.mean(fold_memories)
    })

# 创建 DataFrame 并保存结果

results_df = pd.DataFrame(output_results)
results_df = results_df[["dataset", "mean_accuracy", "mean_runtime", "mean_memory"]]  # 选择关键列

print(results_df)
